# Notebook for causal analysis for the GENDER subgroup using DoWhy-library

In [22]:
from dowhy import CausalModel
import pandas as pd

### Define outcome and the confounders for each feature

In [23]:
outcome = "How happy are you?"

In [24]:
feature_confounder_map = {
    "Health condition": [
        "Age",
        "Income quartiles",
        "Chronic health problems?",
        "Education completed",
        "Employment - 7 groups"
    ],

    "I generally feel that what I do in life is worthwhile": [
        "A person to get support from when feeling depressed",
        "How frequently participate in social activities?",
        "Employment - 7 groups",
        "Marital status",
        "Health condition"
    ],

    "Can't find the way because life has become so complicated?": [
        "Education completed",
        "Employment - 7 groups",
        "A person to get support from when feeling depressed",
        "Age",
        "Household size"
    ],

    "I feel I am free to decide how to live my life": [
        "Income quartiles",
        "Education completed",
        "How much trust the government?"
    ],

    "I am optimistic about the future": [
        "Personal financial situation",
        "Access to recreational or green areas?",
        "A person to get support from to raise emergency money",
        "Health condition",
        "Age",
        "Employment - 7 groups"
    ],

    "Household able to make ends meet?": [
        "Employment - 7 groups",
        "Income quartiles",
        "Personal financial situation",
        "Household size",
        "No. of children"
    ],

    "Personal financial situation": [
        "Employment - 7 groups",
        "Can afford a meal with meat/chicken/fish every second day?",
        "Household structure",
        "Income quartiles",
        "Education completed"
    ],

    "How much trust the police?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the government?",
        "How much trust the legal system?",
        "Rural/urban living",
        "Age"
    ],

    "Deprivation index: No. of items hhold can't afford": [
        "Income quartiles",
        "Employment - 7 groups",
        "Can afford to keep home adequately warm?",
        "Household size",
        "Household structure"
    ],

    "Quality of education system?": [
        "Education completed",
        "How much trust the legal system?",
        "Income quartiles",
        "Rural/urban living",
        "Age"
    ],

    "The value of what I do is not recognised by others?": [
        "Employment - 7 groups",
        "How frequently participate in social activities?",
        "A person to get support from when feeling depressed",
        "Education completed",
        "Marital status"
    ],

    "Marital status": [
        "Age",
        "Household structure",
        "How frequently participate in social activities?",
        "No. of children",
        "Education completed"
    ],

    "Can most people be trusted?": [
        "Neighbourhood problems - crime, violence or vandalism",
        "How much trust the press?",
        "How much trust the government?",
        "Education completed",
        "Rural/urban living"
    ]
}

### Calculate ATE for the two subgroups

In [25]:
# Data
data = pd.read_csv('../data/data_eqls_LOOSE_filtered.csv')

# Reversing the scale on some features (higher values = more positive)
data["Health condition"] = 6 - data["Health condition"]
data["Can't find the way because life has become so complicated?"] = 6 - data["Can't find the way because life has become so complicated?"]
data["Household able to make ends meet?"] = 7 - data["Household able to make ends meet?"]
data["A person to get support from when feeling depressed"] = 3 - data["A person to get support from when feeling depressed"]
data["I feel I am free to decide how to live my life"] = 6 - data["I feel I am free to decide how to live my life"]
data["I generally feel that what I do in life is worthwhile"] = 6 - data["I generally feel that what I do in life is worthwhile"]
data["I am optimistic about the future"] = 6 - data["I am optimistic about the future"]
data["Marital status"] = (data["Marital status"] == 1).astype(int)


def calculate_causal_values(data_source, confounder_map):
    # Save results
    results = []
    # Iterate through all features
    for treatment, confounders in confounder_map.items():
        try:
            print(f"Treatment: {treatment}")
            model = CausalModel(
                data=data_source,
                treatment=treatment,
                outcome=outcome,
                common_causes=confounders
            )

            identified_estimand = model.identify_effect()

            estimate = model.estimate_effect(
                identified_estimand,
                method_name="backdoor.linear_regression"
            )

            results.append({
                "Treatment": treatment,
                "ATE (DML)": estimate.value,
                "Confounders": ", ".join(confounders)
            })
        except Exception as e:
            results.append({
                "Treatment": treatment,
                "ATE (DML)": None,
                "Confounders": ", ".join(confounders),
                "Error": str(e)
            })
    return results

In [26]:
# Select only female participants
df_female = data[data["Gender"] == 2]       # Female
# Results
df_results_female = pd.DataFrame(calculate_causal_values(df_female, feature_confounder_map))
df_results_female.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
display(df_results_female)

Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE (DML),Confounders
0,Health condition,0.697529,"Age, Income quartiles, Chronic health problems..."
11,Marital status,0.633934,"Age, Household structure, How frequently parti..."
1,I generally feel that what I do in life is wor...,0.612476,A person to get support from when feeling depr...
3,I feel I am free to decide how to live my life,0.526695,"Income quartiles, Education completed, How muc..."
6,Personal financial situation,0.505399,"Employment - 7 groups, Can afford a meal with ..."
2,Can't find the way because life has become so ...,-0.449782,"Education completed, Employment - 7 groups, A ..."
10,The value of what I do is not recognised by ot...,0.411935,"Employment - 7 groups, How frequently particip..."
4,I am optimistic about the future,0.399065,"Personal financial situation, Access to recrea..."
5,Household able to make ends meet?,0.350716,"Employment - 7 groups, Income quartiles, Perso..."
8,Deprivation index: No. of items hhold can't af...,-0.294379,"Income quartiles, Employment - 7 groups, Can a..."


In [27]:
# Select only male participants
df_male = data[data["Gender"] == 1]       # Male
df_results_male = pd.DataFrame(calculate_causal_values(df_male, feature_confounder_map))
df_results_male.sort_values(by="ATE (DML)", ascending=False, inplace=True, key=abs)
display(df_results_male)

C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]


,Treatment,ATE (DML),Confounders
0,Health condition,0.608349,"Age, Income quartiles, Chronic health problems..."
1,I generally feel that what I do in life is wor...,0.595794,A person to get support from when feeling depr...
6,Personal financial situation,0.481392,"Employment - 7 groups, Can afford a meal with ..."
3,I feel I am free to decide how to live my life,0.464300,"Income quartiles, Education completed, How muc..."
2,Can't find the way because life has become so ...,-0.433278,"Education completed, Employment - 7 groups, A ..."
4,I am optimistic about the future,0.378026,"Personal financial situation, Access to recrea..."
10,The value of what I do is not recognised by ot...,0.374873,"Employment - 7 groups, How frequently particip..."
8,Deprivation index: No. of items hhold can't af...,-0.366435,"Income quartiles, Employment - 7 groups, Can a..."
11,Marital status,0.351383,"Age, Household structure, How frequently parti..."
5,Household able to make ends meet?,0.330337,"Employment - 7 groups, Income quartiles, Perso..."


### Combined Results

In [28]:
# Female
df_results_female = pd.DataFrame(
    calculate_causal_values(df_female, feature_confounder_map)
).rename(columns={"ATE (DML)": "ATE_female"})

# Male
df_results_male = pd.DataFrame(
    calculate_causal_values(df_male, feature_confounder_map)
).rename(columns={"ATE (DML)": "ATE_male"})


df_compare = pd.merge(
    df_results_female[["Treatment", "ATE_female"]],
    df_results_male[["Treatment", "ATE_male"]],
    on="Treatment",
    how="inner"
)

df_compare = pd.merge(
    df_results_female[["Treatment", "ATE_female"]],
    df_results_male[["Treatment", "ATE_male"]],
    on="Treatment",
    how="inner"
)
df_compare = df_compare.sort_values("ATE_female", ascending=False, key=abs)
display(df_compare)

Treatment: Health condition
Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?
Treatment: Health condition


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

Treatment: I generally feel that what I do in life is worthwhile
Treatment: Can't find the way because life has become so complicated?
Treatment: I feel I am free to decide how to live my life
Treatment: I am optimistic about the future
Treatment: Household able to make ends meet?
Treatment: Personal financial situation
Treatment: How much trust the police?
Treatment: Deprivation index: No. of items hhold can't afford
Treatment: Quality of education system?
Treatment: The value of what I do is not recognised by others?
Treatment: Marital status
Treatment: Can most people be trusted?


C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept_parameter = self.model.params[0]
C:\Users\Minh\anaconda3\envs\happiness_project\Lib\site-packages\dowhy\causal_estimators\regression_estimator.py:131: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future vers

,Treatment,ATE_female,ATE_male
0,Health condition,0.697529,0.608349
11,Marital status,0.633934,0.351383
1,I generally feel that what I do in life is wor...,0.612476,0.595794
3,I feel I am free to decide how to live my life,0.526695,0.464300
6,Personal financial situation,0.505399,0.481392
2,Can't find the way because life has become so ...,-0.449782,-0.433278
10,The value of what I do is not recognised by ot...,0.411935,0.374873
4,I am optimistic about the future,0.399065,0.378026
5,Household able to make ends meet?,0.350716,0.330337
8,Deprivation index: No. of items hhold can't af...,-0.294379,-0.366435


In [29]:
df_compare.to_csv("Results/results_gender_causal_analysis.csv")